In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").option("quote",'"').option("escape",'"').load("/Volumes/Assignment7/assign7/volume7/dataset.csv")

In [0]:
display(df)

In [0]:
from pyspark.sql.functions import col
df=df.select([col(c).alias(c.replace(" ","_")) for c in df.columns])

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("customers")

In [0]:
%sql
select * from customers;

In [0]:
%sql
DESCRIBE DETAIL customers;

In [0]:
df=spark.read.table("customers")

In [0]:
df=df.dropDuplicates()

In [0]:
#checking null values
from pyspark.sql.functions import col,when,count
df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()

In [0]:
#No null values found therefore no null handling is done

In [0]:
#checking schema
df.printSchema()
display(df)

In [0]:
#observing the schema Quantity is intteger it should be double
#converting the datatype of Quantity to double
from pyspark.sql.functions import col
df=df.withColumn("Quantity",col("Quantity").cast("double"))


In [0]:
df.printSchema()

In [0]:
#creating updated records from existing dataset
from pyspark.sql.functions import col
updated_df=(df.limit(80)
            .withColumn("Sales",col("Sales")*1.10)
            .withColumn("Quantity",col("Quantity")+1)
            .withColumn("Discount",col("Discount")+0.01)
            .withColumn("Profit",col("Profit")+100)
            )

In [0]:
#creating new records 
from pyspark.sql.functions import monotonically_increasing_id,lit
new_df=(df.limit(40)
        .withColumn("Row_ID",col("Row_ID")+10000)
        .withColumn("Customer_ID",lit("NEW-001"))
        .withColumn("Customer_Name",lit("Alex Brown"))
        )

In [0]:
incremental_df=updated_df.unionByName(new_df)

In [0]:
incremental_df.write.format("delta").mode("overwrite").saveAsTable("incremental_customers")

In [0]:
%sql
DESCRIBE customers;

In [0]:
%sql
MERGE INTO customers c
USING incremental_customers i
ON c.Row_ID = i.Row_ID
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED
THEN INSERT *;

In [0]:
spark.sql("""select count(*) from customers""").show()

In [0]:
#Checking Duplicate Records
spark.sql("""select Row_ID,count(*) from customers group by Row_ID having count(*)>1""")

In [0]:
#verifying new Record
spark.sql("""select * from customers where Customer_ID='NEW-001' limit 1""").show()


In [0]:
#verifying updated Record
spark.sql("""select * from customers where Row_ID=1""").show()

In [0]:
display(spark.table('customers'))

In [0]:
from pyspark.sql.functions import count
print("Total Records :",
      spark.table("customers").count())
print("Total Columns :",
      len(spark.table("customers").columns))

In [0]:
df=spark.table("customers")
df.write.mode("overwrite").option("header", "true").csv("/Volumes/Assignment7/assign7/volume7/final_dataset.csv")